# 01 — Project Problem And Dataset Selection

**Pipeline role.** First notebook in the eight-notebook pipeline. Defines the business problem, prediction target, scope, success criteria, and dataset provenance for the ISOM3360 Spring 2026 group project.

**Topic.** Predict whether a hotel booking will be cancelled (binary classification on `is_canceled`) using the Hotel Booking Demand dataset.

**Rubric coverage (from `Group Project Guideline.md`).**
- Report Section 1 — Introduction: problem framing, motivation, larger context.
- Report Section 2 — Data Understanding (setup only): dataset source, access notes, target definition.
- Report Section 6 — References: upstream attribution and dataset citation trail.

**TL;DR for teammates.** By the end of this notebook, the project problem, target, evaluation angle, success criteria, and dataset provenance should be fixed. No analysis is performed here; that begins in notebook 02.


## Business Problem And Motivation

Hotels face large revenue and operational costs from last-minute cancellations. A usable cancellation model supports:

- **Revenue management.** Smarter overbooking and discounting policies.
- **Occupancy planning.** More accurate expected arrivals for staffing and inventory.
- **Customer operations.** Targeted interventions (reminders, deposit offers) for high-risk bookings.

This is a well-suited ISOM3360 project because the problem is:

- predictive rather than descriptive,
- backed by a public, real-world dataset of sufficient size (>5,000 and <500,000 rows, >20 features — see rubric),
- solvable with classical supervised learning methods taught in the course,
- and meaningful enough to motivate a written report with genuine business interpretation.

## Prediction Target And Problem Type

- **Target.** `is_canceled` ∈ {0, 1}.
- **Problem type.** Supervised binary classification.
- **Unit of observation.** One booking record.
- **Positive class.** `is_canceled = 1` (cancelled booking). This is the class we want to detect.

## Success Criteria (Working)

- **Technical.** Beat a majority-class baseline on ROC-AUC and F1 by a clearly interpretable margin, with an honest held-out test evaluation.
- **Process.** Every modelling choice (preprocessing, features, algorithm, tuning) is explained and traceable to an experiment-log entry.
- **Deliverables.** A report that maps cleanly onto the six sections of `Group Project Guideline.md` and a notebook pipeline that a reviewer can rerun end-to-end.


## Dataset Provenance And Intake Status

- **Dataset.** Hotel Booking Demand (Antonio, de Almeida, & Nunes, 2019).
- **Upstream attribution.** `@manthangandhi/hotel_cancellation_prediction` — used as the topic and provenance reference for this project. This project is not a clone; it reorganizes the analysis around the ISOM3360 rubric.
- **Local file.** `data/raw/hotel_bookings.csv` (downloaded 2026-04-18).
- **Verified shape at intake.** 119,390 rows × 32 columns; satisfies the rubric size requirement.
- **Detailed provenance notes.** See [../references/dataset_links.md](../references/dataset_links.md).
- **Column-level notes.** See [../references/data_dictionary_template.md](../references/data_dictionary_template.md).

## Scope Boundaries

In scope:
- Classical supervised classification models (logistic regression, decision tree, random forest, gradient boosting, k-NN, naive Bayes).
- Standard preprocessing: imputation, encoding, scaling, outlier handling.
- Cross-validation, hyperparameter tuning, threshold tuning, evaluation with ROC-AUC / F1 / precision / recall / accuracy.

Out of scope (unless later justified):
- Deep learning models.
- External data enrichment.
- Time-series forecasting framing (we use a static classification framing).

## Assumptions And Known Risks

- The dataset reflects historical behaviour that may not generalize to future periods.
- Two known leakage-risk columns (`reservation_status`, `reservation_status_date`) must be excluded from features; two more (`assigned_room_type`, `booking_changes`) need leakage review in notebook 02.
- Class balance is not yet verified; notebook 02 will check and notebook 03/06 will adjust strategy if needed.


## Methodological Influences

Problem framing and documentation style follow ISOM3360 data-preparation practice and the classification-pipeline discipline introduced in course tutorials. See the shared [project conventions](../references/project_conventions.md) for standards applied across all notebooks.


## Inputs, Outputs, Artifacts

**Inputs.**
- `Group Project Guideline.md` (rubric).
- `references/dataset_links.md` (provenance).
- `data/raw/hotel_bookings.csv` (intake file).

**Outputs of this notebook.**
- A finalized problem statement paragraph reusable in the report's Introduction.
- A finalized target definition and success criteria paragraph.
- A confirmed dataset intake record.

**Downstream consumers.**
- Notebook 02 uses the target definition and scope to plan EDA.
- Notebook 08 pulls the problem statement and motivation into the report assembly.


## Implementation Block 1 — Local Intake Verification

This is the first practical block for Notebook 01. It keeps the notebook aligned with [../Group Project Guideline.md](../Group%20Project%20Guideline.md) by verifying that the local dataset satisfies the project-size requirement before any downstream work begins.

This block should only verify intake and set stable project-level variables. It should not perform EDA, missing-value analysis, or target-distribution analysis; those belong in Notebook 02.


In [1]:
from pathlib import Path

import pandas as pd

RAW_DATA_PATH = Path("../data/raw/hotel_bookings.csv")
TARGET_COLUMN = "is_canceled"
PROJECT_TOPIC = "Hotel cancellation prediction"
UPSTREAM_ATTRIBUTION = "@manthangandhi/hotel_cancellation_prediction"
MIN_ROWS_REQUIRED = 5_000
MAX_ROWS_ALLOWED = 500_000
MIN_FEATURES_REQUIRED = 20

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Expected raw dataset at {RAW_DATA_PATH.resolve()}, but the file was not found."
    )

hotel_bookings = pd.read_csv(RAW_DATA_PATH)
row_count, column_count = hotel_bookings.shape
feature_count_before_modeling = column_count - 1

project_context = {
    "project_topic": PROJECT_TOPIC,
    "target_column": TARGET_COLUMN,
    "raw_data_path": str(RAW_DATA_PATH),
    "row_count": row_count,
    "column_count": column_count,
    "feature_count_before_modeling": feature_count_before_modeling,
    "meets_row_requirement": MIN_ROWS_REQUIRED < row_count < MAX_ROWS_ALLOWED,
    "meets_feature_requirement": feature_count_before_modeling > MIN_FEATURES_REQUIRED,
    "upstream_attribution": UPSTREAM_ATTRIBUTION,
}

if TARGET_COLUMN not in hotel_bookings.columns:
    raise KeyError(f"Target column '{TARGET_COLUMN}' was not found in the dataset.")

project_intake_summary = pd.Series(project_context, name="value")
project_intake_summary


project_topic                                   Hotel cancellation prediction
target_column                                                     is_canceled
raw_data_path                                  ../data/raw/hotel_bookings.csv
row_count                                                              119390
column_count                                                               32
feature_count_before_modeling                                              31
meets_row_requirement                                                    True
meets_feature_requirement                                                True
upstream_attribution             @manthangandhi/hotel_cancellation_prediction
Name: value, dtype: object

## Implementation Block 2 — Report-Ready Project Brief

This block converts the narrative framing above into stable, reusable project metadata for the later notebooks and the final report.

- [../Group Project Guideline.md](../Group%20Project%20Guideline.md) — Introduction, Model Building, and Performance Evaluation sections.

This is still a Notebook 01 framing block. It should define project intent and working success criteria, not produce model results.


In [4]:
project_brief = {
    "business_problem": (
        "Hotel cancellations reduce occupancy predictability and can hurt revenue, staffing, "
        "and operational planning."
    ),
    "primary_decision_supported": (
        "Identify bookings with elevated cancellation risk early enough to support "
        "overbooking, pricing, and customer follow-up decisions."
    ),
    "primary_user": "Hotel revenue or operations manager",
    "model_task": "Binary classification",
    "target_column": TARGET_COLUMN,
    "positive_class_meaning": "1 = booking cancelled",
    "unit_of_observation": "One hotel booking record",
    "dataset_source": "Hotel Booking Demand dataset",
    "upstream_attribution": UPSTREAM_ATTRIBUTION,
}

working_success_criteria = {
    "technical_goal": (
        "Beat a majority-class baseline on ROC-AUC and F1 using an honest held-out test set."
    ),
    "process_goal": (
        "Keep preprocessing, feature engineering, model choice, and tuning decisions traceable "
        "through notebook narrative and the experiment log."
    ),
    "report_goal": (
        "Produce artifacts that map cleanly to the six report sections in Group Project Guideline.md."
    ),
    "primary_metric": "ROC-AUC",
    "supporting_metrics": "F1, precision, recall, accuracy",
    "default_test_strategy": "Stratified held-out test set fixed before modeling",
}

project_brief_table = pd.DataFrame(
    {"item": list(project_brief.keys()), "value": list(project_brief.values())}
)
working_success_criteria_table = pd.DataFrame(
    {
        "item": list(working_success_criteria.keys()),
        "value": list(working_success_criteria.values()),
    }
)

project_brief_table


,item,value
0,business_problem,Hotel cancellations reduce occupancy predictab...
1,primary_decision_supported,Identify bookings with elevated cancellation r...
2,primary_user,Hotel revenue or operations manager
3,model_task,Binary classification
4,target_column,is_canceled
5,positive_class_meaning,1 = booking cancelled
6,unit_of_observation,One hotel booking record
7,dataset_source,Hotel Booking Demand dataset
8,upstream_attribution,@manthangandhi/hotel_cancellation_prediction


## Implementation Block 3 — Draft Introduction Paragraph And Stakeholder Framing

This block turns the structured project metadata into reusable report language for the final submission.

It stays within Notebook 01 scope by drafting communication-ready text rather than performing analysis. The block is aligned with:
- [../Group Project Guideline.md](../Group%20Project%20Guideline.md) — the report needs a motivated Introduction and a clear explanation of why the problem matters.

The goal here is not to finalize polished prose forever. It is to give the team a stable first-draft introduction and stakeholder summary that later iterations can refine.


In [6]:
report_intro_paragraph = (
    f"{project_brief['business_problem']} This project addresses that issue as a "
    f"{project_brief['model_task'].lower()} task using the {project_brief['dataset_source']} "
    f"with {row_count:,} booking records and {column_count} columns. The goal is to predict whether "
    f"a booking will be cancelled ({project_brief['positive_class_meaning']}) so that "
    f"{project_brief['primary_user'].lower()}s can make earlier and better-informed operational and "
    f"revenue decisions. The project topic and dataset provenance are tracked with attribution to "
    f"{project_brief['upstream_attribution']}, while the final workflow and deliverables are organized "
    f"to match the ISOM3360 group project guideline."
)

stakeholder_use_case_summary = pd.DataFrame(
    [
        {
            "stakeholder": project_brief["primary_user"],
            "decision_supported": project_brief["primary_decision_supported"],
            "business_value": "Better occupancy planning, overbooking control, and customer follow-up.",
            "primary_metric": working_success_criteria["primary_metric"],
            "supporting_metrics": working_success_criteria["supporting_metrics"],
        }
    ]
)

report_intro_paragraph


'Hotel cancellations reduce occupancy predictability and can hurt revenue, staffing, and operational planning. This project addresses that issue as a binary classification task using the Hotel Booking Demand dataset with 119,390 booking records and 32 columns. The goal is to predict whether a booking will be cancelled (1 = booking cancelled) so that hotel revenue or operations managers can make earlier and better-informed operational and revenue decisions. The project topic and dataset provenance are tracked with attribution to @manthangandhi/hotel_cancellation_prediction, while the final workflow and deliverables are organized to match the ISOM3360 group project guideline.'

In [7]:
project_scope_summary = pd.DataFrame(
    [
        {
            "topic": PROJECT_TOPIC,
            "target": TARGET_COLUMN,
            "task": project_brief["model_task"],
            "primary_metric": working_success_criteria["primary_metric"],
            "supporting_metrics": working_success_criteria["supporting_metrics"],
            "test_strategy": working_success_criteria["default_test_strategy"],
            "known_leakage_review": "reservation_status, reservation_status_date, assigned_room_type, booking_changes",
            "upstream_attribution": UPSTREAM_ATTRIBUTION,
        }
    ]
)

team_review_snapshot = {
    "project_brief": project_brief_table,
    "success_criteria": working_success_criteria_table,
    "stakeholder_summary": stakeholder_use_case_summary,
    "scope_summary": project_scope_summary,
}

team_review_snapshot["scope_summary"]


,topic,target,task,primary_metric,supporting_metrics,test_strategy,known_leakage_review,upstream_attribution
0,Hotel cancellation prediction,is_canceled,Binary classification,ROC-AUC,"F1, precision, recall, accuracy",Stratified held-out test set fixed before mode...,"reservation_status, reservation_status_date, a...",@manthangandhi/hotel_cancellation_prediction


## Implementation Block 4 — Team Review Snapshot

This block makes the existing project metadata easier to review during team meetings and before work starts in Notebook 02.

The emphasis here is readability and reuse, not new analysis. This aligns with:
- [../Group Project Guideline.md](../Group%20Project%20Guideline.md) — the project should show clear progress, rationale, and organization.

This snapshot should help teammates quickly confirm the project objective, evaluation focus, and scope boundaries before the workflow moves into dataset understanding.


## Implementation Block 5 — Review Tables For Team Check-In

This block exposes the working success criteria and stakeholder framing as compact tables that teammates can review without reading through all earlier cells.

It remains within Notebook 01 scope by improving reviewability rather than adding any new modelling or data-understanding work.


In [9]:
from IPython.display import display

review_tables = {
    "success_criteria": working_success_criteria_table.copy(),
    "stakeholder_summary": stakeholder_use_case_summary.copy(),
}

display(review_tables["success_criteria"])
display(review_tables["stakeholder_summary"])


,item,value
0,technical_goal,Beat a majority-class baseline on ROC-AUC and ...
1,process_goal,"Keep preprocessing, feature engineering, model..."
2,report_goal,Produce artifacts that map cleanly to the six ...
3,primary_metric,ROC-AUC
4,supporting_metrics,"F1, precision, recall, accuracy"
5,default_test_strategy,Stratified held-out test set fixed before mode...


,stakeholder,decision_supported,business_value,primary_metric,supporting_metrics
0,Hotel revenue or operations manager,Identify bookings with elevated cancellation r...,"Better occupancy planning, overbooking control...",ROC-AUC,"F1, precision, recall, accuracy"


## Key Questions To Answer Here

1. What decision does this model support, and for whom?
2. Why is a classical ML approach appropriate for this problem?
3. What exactly does `is_canceled = 1` mean operationally?
4. What quantitative bar must the final model clear to be called "useful"?
5. Which columns are already known to leak the outcome?
6. What is the attribution trail from source → local file → report?

## Remaining Human Review Items

- [ ] Refine `report_intro_paragraph` into the final wording for the written report.
- [ ] Decide whether to keep the current working success criteria as-is or tighten them with an explicit numeric target.
- [ ] Confirm the preferred stakeholder wording for the report (for example, revenue manager only vs revenue and operations managers).
- [ ] Carry the locked problem framing and leakage notes forward unchanged into Notebook 02.


## Review Checklist

- [x] Problem statement is specific, predictive, and socially/commercially meaningful (rubric §1).
- [x] Target variable is unambiguously defined.
- [x] Success criteria are stated in terms that downstream evaluation can meet.
- [x] Dataset provenance is recorded with file name, size, columns, access date.
- [x] Upstream attribution to `@manthangandhi/hotel_cancellation_prediction` is present.
- [x] Leakage-risk columns are flagged for notebook 02 follow-up.
- [x] Scope boundaries are explicit (what this project will and will not do).
- [x] Team-review artifacts are visible in notebook outputs (project brief, success criteria, stakeholder summary, scope summary).

## Handoff To Notebook 02

Notebook 02 (`02_data_understanding.ipynb`) starts from the confirmed intake record and produces schema, missingness, target-distribution, outlier, and leakage-review evidence required by rubric §2. The framing, target definition, and leakage cautions established here should be treated as locked inputs unless Notebook 02 uncovers a documented contradiction in the raw data.
